In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
BOLD = "\033[1m"
CYAN = "\033[36m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
MAG = "\033[95m"
RESET = "\033[0m"
RED = "\033[91m"

In [3]:
import yaml
from src.data.loader import load_dataset
from src.evaluation.benchmark import run_dataset_benchmark
from src.models.base import BaseForecaster

In [4]:
MODEL_NAMES = ["chronos2", "moirai2", "timesfm3", "seasonal_naive"]
DATASET = "all"
WINDOWS = None
SAVE_FORECASTS = True

In [5]:
BENCHMARK_CONFIG = PROJECT_ROOT / "configs" / "benchmark.yaml"
TARGET_CONFIG = PROJECT_ROOT / "configs" / "benchmark_targets.yaml"

In [12]:
def create_forecaster(
        model_name: str,
        seasonal_period: int | None = None
) -> BaseForecaster:
    if model_name == 'chronos2':
        from src.models.chronos import Chronos2Forecaster
        return Chronos2Forecaster()
    if model_name == 'moirai2':
        from src.models.moirai import Moirai2Forecaster
        return Moirai2Forecaster()
    if model_name == 'timesfm3':
        from src.models.timesfm import TimesFM3Forecaster
        return TimesFM3Forecaster()
    if model_name == "seasonal_naive":
        from src.models.baselines import SeasonalNaiveForecaster
        if seasonal_period is None:
            raise ValueError("seasonal_period is required for Seasonal Naive.")
        return SeasonalNaiveForecaster(seasonal_period=seasonal_period)
    raise ValueError(f"Unknown model: {model_name}")

In [13]:
def load_configs() -> tuple[dict, dict]: 
    with open(BENCHMARK_CONFIG, "r", encoding="utf-8") as f:
        benchmark = yaml.safe_load(f)["benchmark"]
    print(f"Successfully loaded {BOLD}{YELLOW}{BENCHMARK_CONFIG.relative_to(PROJECT_ROOT).as_posix()}{RESET}")
    
    with open(TARGET_CONFIG, "r", encoding="utf-8") as f:
        targets = yaml.safe_load(f)["targets"]
    print(f"Successfully loaded {BOLD}{YELLOW}{TARGET_CONFIG.relative_to(PROJECT_ROOT).as_posix()}")
    
    return benchmark, targets

In [14]:
benchmark, targets = load_configs()

Successfully loaded configs/benchmark.yaml
Successfully loaded configs/benchmark_targets.yaml


In [15]:
if DATASET == 'all':
    datasets = benchmark['datasets']
    print(f"{BOLD}{MAG}Running with all datasets{RESET}.")
    print("+", datasets)
else:
    if DATASET not in benchmark['datasets']:
        raise ValueError(f"{BOLD}{RED}{DATASET} is not part of the benchmark.{RESET}")
    datasets = [DATASET]

Running with all datasets.
+ ['ETTh1', 'Weather', 'Electricity', 'Traffic', 'Exchange', 'Solar']


In [16]:
n_windows = WINDOWS or benchmark["windows_per_target"]

In [18]:
shared_forecaster = None
def run(model_name):
    if model_name != "seasonal_naive":
        shared_forecaster = create_forecaster(model_name)
    print(f"--- {BOLD}{YELLOW}{model_name.replace('_', ' ').title()}{RESET} --------------------------------------------------")
    for dataset_name in datasets:
        print(f"Evaluating on {BOLD}{MAG}{dataset_name}{RESET}...")
        dataset = load_dataset(dataset_name)
        
        if model_name == 'seasonal_naive':
            forecaster = create_forecaster(model_name=model_name, seasonal_period=dataset.seasonal_period)
        else:
            forecaster = shared_forecaster
            
        run_dataset_benchmark(
            forecaster=forecaster,
            dataset_name=dataset_name,
            targets=targets[dataset_name],
            context_length=benchmark["context_length"],
            prediction_lengths=benchmark["prediction_lengths"],
            n_windows=n_windows,
            save_forecasts=SAVE_FORECASTS)
    print()

In [19]:
run(MODEL_NAMES[0])

Loading Chronos-2 from amazon/chronos-2 on cuda
Chronos-2 loaded
--- Chronos2 --------------------------------------------------
Evaluating on ETTh1...

Chronos2Forecaster | ETTh1 | OT | H=24 | 20 windows ----------
  + Window 00 | Target: OT | MASE: 0.3020 | Time: 0.05s
  + Window 01 | Target: OT | MASE: 0.7255 | Time: 0.05s
  + Window 02 | Target: OT | MASE: 1.7345 | Time: 0.05s
  + Window 03 | Target: OT | MASE: 0.8535 | Time: 0.05s
  + Window 04 | Target: OT | MASE: 0.5577 | Time: 0.05s
  + Window 05 | Target: OT | MASE: 0.6168 | Time: 0.03s
  + Window 06 | Target: OT | MASE: 0.5676 | Time: 0.03s
  + Window 07 | Target: OT | MASE: 0.4202 | Time: 0.03s
  + Window 08 | Target: OT | MASE: 0.2937 | Time: 0.03s
  + Window 09 | Target: OT | MASE: 0.7472 | Time: 0.03s
  + Window 10 | Target: OT | MASE: 0.5423 | Time: 0.03s
  + Window 11 | Target: OT | MASE: 0.3520 | Time: 0.03s
  + Window 12 | Target: OT | MASE: 1.9726 | Time: 0.03s
  + Window 13 | Target: OT | MASE: 1.3059 | Time: 0.03s


In [20]:
run(MODEL_NAMES[1])

Loading Moirai-2 from Salesforce/moirai-2.0-R-small on cuda
Moirai-2 loaded
Quantiles: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
--- Moirai2 --------------------------------------------------
Evaluating on ETTh1...

Moirai2Forecaster | ETTh1 | OT | H=24 | 20 windows ----------
  + Window 00 | Target: OT | MASE: 0.3793 | Time: 0.17s
  + Window 01 | Target: OT | MASE: 0.6889 | Time: 0.03s
  + Window 02 | Target: OT | MASE: 1.7620 | Time: 0.03s
  + Window 03 | Target: OT | MASE: 0.5832 | Time: 0.03s
  + Window 04 | Target: OT | MASE: 0.4894 | Time: 0.03s
  + Window 05 | Target: OT | MASE: 0.5531 | Time: 0.03s
  + Window 06 | Target: OT | MASE: 0.3922 | Time: 0.03s
  + Window 07 | Target: OT | MASE: 0.2811 | Time: 0.03s
  + Window 08 | Target: OT | MASE: 0.3328 | Time: 0.03s
  + Window 09 | Target: OT | MASE: 1.0200 | Time: 0.03s
  + Window 10 | Target: OT | MASE: 0.5728 | Time: 0.02s
  + Window 11 | Target: OT | MASE: 0.3287 | Time: 0.02s
  + Window 12 | Target: OT | MASE: 2.0442 | Ti

In [21]:
run(MODEL_NAMES[2])

Loading TimesFM 3.0 from google/timesfm-3.0-pytorch on cuda...
TimesFM 3.0 loaded.
--- Timesfm3 --------------------------------------------------
Evaluating on ETTh1...

TimesFM3Forecaster | ETTh1 | OT | H=24 | 20 windows ----------
  + Window 00 | Target: OT | MASE: 0.4182 | Time: 0.39s
  + Window 01 | Target: OT | MASE: 0.7724 | Time: 0.25s
  + Window 02 | Target: OT | MASE: 1.8441 | Time: 0.26s
  + Window 03 | Target: OT | MASE: 0.7338 | Time: 0.25s
  + Window 04 | Target: OT | MASE: 0.5891 | Time: 0.25s
  + Window 05 | Target: OT | MASE: 0.7249 | Time: 0.25s
  + Window 06 | Target: OT | MASE: 0.7882 | Time: 0.33s
  + Window 07 | Target: OT | MASE: 0.3054 | Time: 0.28s
  + Window 08 | Target: OT | MASE: 0.2953 | Time: 0.30s
  + Window 09 | Target: OT | MASE: 0.8095 | Time: 0.28s
  + Window 10 | Target: OT | MASE: 0.6476 | Time: 0.29s
  + Window 11 | Target: OT | MASE: 0.3462 | Time: 0.27s
  + Window 12 | Target: OT | MASE: 1.9325 | Time: 0.31s
  + Window 13 | Target: OT | MASE: 1.1

In [22]:
run(MODEL_NAMES[3])

--- Seasonal Naive --------------------------------------------------
Evaluating on ETTh1...

SeasonalNaiveForecaster | ETTh1 | OT | H=24 | 20 windows ----------
  + Window 00 | Target: OT | MASE: 0.8889 | Time: 0.00s
  + Window 01 | Target: OT | MASE: 0.8870 | Time: 0.00s
  + Window 02 | Target: OT | MASE: 2.5342 | Time: 0.00s
  + Window 03 | Target: OT | MASE: 1.8644 | Time: 0.00s
  + Window 04 | Target: OT | MASE: 1.0659 | Time: 0.00s
  + Window 05 | Target: OT | MASE: 1.3227 | Time: 0.00s
  + Window 06 | Target: OT | MASE: 0.7777 | Time: 0.00s
  + Window 07 | Target: OT | MASE: 0.3345 | Time: 0.00s
  + Window 08 | Target: OT | MASE: 0.5908 | Time: 0.00s
  + Window 09 | Target: OT | MASE: 1.4484 | Time: 0.00s
  + Window 10 | Target: OT | MASE: 0.8784 | Time: 0.00s
  + Window 11 | Target: OT | MASE: 0.9658 | Time: 0.00s
  + Window 12 | Target: OT | MASE: 1.6414 | Time: 0.00s
  + Window 13 | Target: OT | MASE: 2.3187 | Time: 0.00s
  + Window 14 | Target: OT | MASE: 1.0734 | Time: 0.00